In [1]:
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, load_training_checkpoint
import mlflow
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKHybridMamba2Adapter
from model_classes import WaitKHybridMamba2MT, SimulHybridMamba2Config

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_hybrid_distillation")

from transformers import AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [3]:
hybrid_config = SimulHybridMamba2Config(
    vocab_size = len(tokenizer),
    d_model = 512,
    num_encoder_layers = 6,
    num_decoder_layers = 6,
    d_state = 128,
    d_conv = 4,
    expand = 2,
    headdim = 64,
    ngroups = 1,
    nhead = 8,
    dim_feedforward = 2048,
    dropout = 0.1,
    max_source_len = 64,
    max_target_len = 64,
    pad_token_id = tokenizer.pad_token_id,
    eos_token_id = tokenizer.eos_token_id,
)

In [4]:
hybrid_mamba2 = WaitKHybridMamba2MT(hybrid_config)

In [5]:
count_parameters(hybrid_mamba2)

Total parameters:     301,988,416
Trainable parameters: 301,988,416
Frozen parameters:    0


{'total': 301988416, 'trainable': 301988416, 'frozen': 0}

In [6]:
train_cfg = TrainConfig(
    epochs=10,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=2e-5,
    use_amp=True,
)

In [7]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [9]:
train_waitk_student(
    student=hybrid_mamba2,
    train_dataset=dataset,
    model_cfg=hybrid_config,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="hybrid",
    resume_from_checkpoint="./checkpoints/hybrid_step_12000.pt"
)

epoch 8/10:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 9/10:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 10/10:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/3/runs/b76a5c8a73b64b7d9899cd1ce49d9144
🧪 View experiment at: http://localhost:5000/#/experiments/3


KeyboardInterrupt: 

In [4]:
hybrid_mamba2 = load_training_checkpoint("./checkpoints/hybrid_epoch_9.pt", SimulHybridMamba2Config, WaitKHybridMamba2MT)[0]

In [5]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

adapter = WaitKHybridMamba2Adapter(
    model=hybrid_mamba2,
    tokenizer=tokenizer,
    name="hybrid_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/hybrid_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Validating hybrid_wait_k, wait_k=3:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 32.23075330505572, 'chrF++': 54.499283690741926, 'TER': 61.479870501962054, 'AP': 0.6645900322666223, 'AL': 4.475102289445078, 'DAL': 5.225846177727018, 'LAAL': 4.774298562475839, 'ATD_text': 3.1745753019531127, 'total_time_sec': 501.29456542399566, 'ms_per_sentence': 1.6201627789146946, 'target_tokens_per_sec': 17874.973355079343, 'source_tokens_per_sec': 15526.324314760735, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 5286.4970703125, 'generation_total_time_sec': 460.76634810102405, 'generation_ms_per_sentence': 1.489177299056346, 'generation_target_tokens_per_sec': 19447.225338677214}


Validating hybrid_wait_k, wait_k=5:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 34.451777380053656, 'chrF++': 56.16760712764305, 'TER': 58.27140912530797, 'AP': 0.7305274362183884, 'AL': 6.32022045709876, 'DAL': 7.0995701448791335, 'LAAL': 6.1824544763869085, 'ATD_text': 4.451102953246315, 'total_time_sec': 505.1873632679999, 'ms_per_sentence': 1.632744136479105, 'target_tokens_per_sec': 17555.22335837843, 'source_tokens_per_sec': 15406.68386804246, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 5286.4970703125, 'generation_total_time_sec': 459.95882217303733, 'generation_ms_per_sentence': 1.4865674094988441, 'generation_target_tokens_per_sec': 19281.458627319444}


Validating hybrid_wait_k, wait_k=7:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 35.039390589296055, 'chrF++': 56.671057567051676, 'TER': 57.33772246897342, 'AP': 0.786327213001359, 'AL': 8.13803584202057, 'DAL': 8.926129812094912, 'LAAL': 7.441594417378618, 'ATD_text': 5.658090285368608, 'total_time_sec': 507.46870880299684, 'ms_per_sentence': 1.6401173485116733, 'target_tokens_per_sec': 17431.541780902335, 'source_tokens_per_sec': 15337.422514895437, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 5286.4970703125, 'generation_total_time_sec': 459.96464307702263, 'generation_ms_per_sentence': 1.486586222413699, 'generation_target_tokens_per_sec': 19231.830387708113}


Validating hybrid_wait_k, wait_k=9:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 35.20622573038628, 'chrF++': 56.81140171947199, 'TER': 57.04033153045114, 'AP': 0.8311857994634554, 'AL': 9.896898297704976, 'DAL': 10.672374056706378, 'LAAL': 8.535203538336447, 'ATD_text': 6.723756167942194, 'total_time_sec': 508.73524195799837, 'ms_per_sentence': 1.6442107299634736, 'target_tokens_per_sec': 17369.494525267324, 'source_tokens_per_sec': 15299.238892992975, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 5286.4970703125, 'generation_total_time_sec': 459.89650973002426, 'generation_ms_per_sentence': 1.486366018325278, 'generation_target_tokens_per_sec': 19214.04884152595}


Validating hybrid_wait_k, wait_k=11:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 35.257363990256536, 'chrF++': 56.8460657804199, 'TER': 56.920867414415476, 'AP': 0.8669448546912264, 'AL': 11.573429175260452, 'DAL': 12.318438984460718, 'LAAL': 9.473361651385325, 'ATD_text': 7.644876233497969, 'total_time_sec': 509.60032522399706, 'ms_per_sentence': 1.6470066423968102, 'target_tokens_per_sec': 17329.070573332814, 'source_tokens_per_sec': 15273.267332745192, 'first_token_latency_sec': None, 'peak_gpu_memory_mb': 5281.9697265625, 'generation_total_time_sec': 459.7719595479648, 'generation_ms_per_sentence': 1.4859634774181985, 'generation_target_tokens_per_sec': 19207.13044066954}
